In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [2]:
from rag_helper import RAGBase
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [3]:
answer = assistant.rag("How can i run olama locally")
print(answer)

I don't know.


In [4]:
def search(query):
        boost_dict = {"question": 3.0, "section": 0.5}
        filter_dict = {"course": 'llm-zoomcamp'}

        return sqlite_index.search(
            query,
            num_results=5,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

search("How to i install ollama")

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Introduction to LLMs and RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npi

In [55]:
messages = [
    {"role": "user", "content": "How to install ollama?"}
]

response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
)

response.output_text

'## Installing **Ollama** (the local LLM runtime)\n\nOllama provides a simple way to run large language models on your own machine (or a server). Below are step‑by‑step instructions for the three major platforms—macOS, Windows, and Linux—as well as a quick Docker alternative.\n\n> **NOTE:** Ollama requires a fairly recent CPU (AVX2\u202f+\u202fFMA) and at least 8\u202fGB of RAM for the smallest models. Larger models (e.g., Llama‑2‑13B) need more memory (16\u202fGB\u202f+\u202fand often a GPU). Make sure you meet the hardware requirements before you start.\n\n---\n\n## 1. macOS (Apple Silicon *or* Intel)\n\n### 1.1. Using the official installer (recommended)\n\n1. Open Terminal.\n2. Run the one‑liner installer:\n\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\n3. The script will:\n   - Detect your architecture (`arm64` for Apple Silicon, `x86_64` for Intel).\n   - Download the appropriate `.pkg` (Apple Silicon) or `.dmg` (Intel) file.\n   - Install the `ollama` binary in

In [56]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [57]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
    tool_choice="required"
)

response.output

[ResponseReasoningItem(id='resp_01ktybec5xf73bxj4qej19y9c6', summary=[], type='reasoning', content=[Content(text='The user asks "How to install ollama?" They likely want steps to install Ollama, an open-source LLM local inference server. Provide instructions for various OS: macOS, Linux, Windows (via WSL or native). Could also include Docker. Provide command line steps, system requirements, verification. Also maybe mention FAQ. Could also search FAQ. Let\'s search.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"install ollama"}', call_id='fc_decebf12-691e-47e6-b332-c8cfe22a0bea', name='search', type='function_call', id='fc_decebf12-691e-47e6-b332-c8cfe22a0bea', namespace=None, status='completed')]

In [58]:
import json

call = response.output[1]
print(call)
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

ResponseFunctionToolCall(arguments='{"query":"install ollama"}', call_id='fc_decebf12-691e-47e6-b332-c8cfe22a0bea', name='search', type='function_call', id='fc_decebf12-691e-47e6-b332-c8cfe22a0bea', namespace=None, status='completed')


In [59]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [60]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
    # tool_choice="required"
)
response.output_text

'**How to install\u202fOllama**\n\n1. **Download the installer** – go to\u202f[https://ollama.com/download](https://ollama.com/download) and pick your OS.\n\n   | OS | What to download | Installation command |\n   |----|------------------|----------------------|\n   | **macOS** | `ollama‑x.x.x.pkg` | Double‑click the\u202f`.pkg` file and follow the macOS installer wizard. |\n   | **Windows** | `ollama‑x.x.x.msi` | Double‑click the\u202f`.msi` file and run the Windows installer. |\n   | **Linux** | – | Run the one‑liner in a terminal: <br>```bash<br>curl -fsSL https://ollama.com/install.sh \\| sh<br>``` |\n\n2. **Verify the installation**  \n   Open a new terminal (or Command\u202fPrompt on Windows) and run:\n\n   ```bash\n   ollama run llama3\n   ```\n\n   * This will download the LLaMA\u202f3 model (~4\u202fGB) the first time and start a local chat‑like interface.\n\n3. **Check that the local server is up**  \n\n   ```bash\n   curl http://localhost:11434\n   ```\n\n   You should see a

In [51]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(725, 377)

In [61]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176
